# 20 Routine Quick Job Cleanup Template

Read-only planning template for routine cleanup and repair jobs.


In [ ]:
import logging
import warnings

logging.getLogger("datajoint").setLevel(logging.WARNING)
warnings.filterwarnings("ignore", message="pkg_resources is deprecated as an API.*", category=UserWarning)

from adamacs.notebook_runtime import bootstrap_ingest_notebook

ctx = bootstrap_ingest_notebook(verbose=False)
repo_root = ctx.repo_root

import datajoint as dj


In [ ]:
import os

# ADAMACS notebook overrides (edit here for local runs).
os.environ.setdefault("ADAMACS_MAX_JOB_ROWS", "50")
os.environ.setdefault("ADAMACS_INITIALS", "NK")
os.environ.setdefault("ADAMACS_DATE_FROM", "2025-01-01")

print("ADAMACS_MAX_JOB_ROWS =", os.environ["ADAMACS_MAX_JOB_ROWS"])
print("ADAMACS_INITIALS     =", os.environ["ADAMACS_INITIALS"])
print("ADAMACS_DATE_FROM    =", os.environ["ADAMACS_DATE_FROM"])


In [ ]:

import pandas as pd

from adamacs.pipeline import (
    subject,
    session,
    scan,
    event,
    trial,
    imaging,
    behavior,
    model,
    denoising,
)

ALLOW_DB_WRITES = False
MAX_ROWS = int(os.environ.get("ADAMACS_MAX_JOB_ROWS", "50"))
INITIALS = os.environ.get("ADAMACS_INITIALS", "NK")
DATE_FROM = os.environ.get("ADAMACS_DATE_FROM", "2025-01-01")

print("ALLOW_DB_WRITES:", ALLOW_DB_WRITES)
print("INITIALS:", INITIALS)
print("DATE_FROM:", DATE_FROM)



## 1) Job overview by schema

Fast check of `status` counts in the key operational schemas.


In [ ]:

schema_map = {
    "imaging": imaging.schema,
    "model": model.schema,
    "denoising": denoising.schema,
}

status_frames = []
for name, schema in schema_map.items():
    try:
        jobs = schema.jobs
        frame = dj.U("status").aggr(jobs, n="count(*)").fetch(format="frame").reset_index()
        frame.insert(0, "schema", name)
        status_frames.append(frame)
    except Exception as exc:
        status_frames.append(pd.DataFrame([{"schema": name, "status": "error", "n": f"unavailable: {exc}"}]))

pd.concat(status_frames, ignore_index=True)



## 2) Recent error jobs (read-only triage)


In [ ]:

def recent_jobs(schema, status="error", limit=50):
    rows = []
    try:
        query = schema.jobs & f'status="{status}"'
        fetched = query.fetch(
            "timestamp",
            "status",
            "host",
            "key_hash",
            "error_message",
            "error_stack",
            "key",
            as_dict=True,
            order_by="timestamp desc",
            limit=limit,
        )
        for row in fetched:
            rows.append(
                {
                    "timestamp": row.get("timestamp"),
                    "status": row.get("status"),
                    "host": row.get("host"),
                    "key_hash": row.get("key_hash"),
                    "error_message": str(row.get("error_message", ""))[:180],
                    "key_preview": str(row.get("key", {}))[:180],
                    "stack_preview": str(row.get("error_stack", ""))[:220],
                }
            )
    except Exception as exc:
        rows.append({"error": f"Could not fetch {status} jobs: {exc}"})
    return pd.DataFrame(rows)

for schema_name, schema in schema_map.items():
    print(f"
### {schema_name}")
    display(recent_jobs(schema, status="error", limit=MAX_ROWS))



## 3) Stranded task snapshots

`task AND NOT populated output` across core pipelines.


In [ ]:

stranded_queries = {
    "imaging": imaging.ProcessingTask & dj.Not(imaging.Processing),
    "dlc_pose": model.PoseEstimationTaskNew & dj.Not(model.PoseEstimationNew),
    "denoising": denoising.DenoisingTask & dj.Not(denoising.Denoising),
}

for name, query in stranded_queries.items():
    try:
        print(f"{name}: {len(query)}")
    except Exception as exc:
        print(f"{name}: unavailable ({exc})")



## 4) Candidate key selection (user/date filter)


In [ ]:

user_key = (session.SessionUser * subject.User & f'initials = "{INITIALS}"').fetch("KEY")
time_key = (session.Session & f'session_datetime >= "{DATE_FROM}"').fetch("KEY")

candidate_keys = (session.Session & user_key & time_key).fetch("KEY")
print("candidate sessions:", len(candidate_keys))

candidate_df = pd.DataFrame(candidate_keys)
candidate_df.head(20)


In [ ]:

output_csv = repo_root / "notebooks" / "tmp_cleanup_candidate_keys.csv"
pd.DataFrame(candidate_keys).to_csv(output_csv, index=False)
print(f"Wrote candidate key preview to {output_csv}")



## 5) Optional write block (disabled)

Keep this disabled for normal use. The examples below are intentionally commented.


In [ ]:

def require_write_mode():
    if not ALLOW_DB_WRITES:
        raise RuntimeError(
            "Database write mode is disabled. Set ALLOW_DB_WRITES=True only in controlled maintenance windows."
        )

# Example cleanup snippets (DO NOT RUN unless explicitly approved):
# require_write_mode()
# cleanup_key = {"session_id": "sessXXXX", "scan_id": "scanXXXX"}
# (imaging.schema.jobs & 'status="error"' & cleanup_key).delete()
# (model.schema.jobs & 'status="error"' & cleanup_key).delete()
# (imaging.ProcessingTask & cleanup_key & dj.Not(imaging.Processing)).delete()
